# Chapter 50 — ChromaDB vs FAISS: Benchmarking Vector Search

In this standalone, code-heavy chapter, you'll store the same embeddings in ChromaDB and FAISS, compare search results, benchmark latency, and validate the functions. Run cells from top to bottom; no earlier chapter files are required.

# 1. Install Dependencies

In [ ]:
%pip install -q chromadb faiss-cpu sentence-transformers numpy pandas

# 2. Import 

In [3]:
import time
import statistics
import numpy as np
import pandas as pd
import chromadb
import faiss
from sentence_transformers import SentenceTransformer

In [4]:
np.random.seed(42)


# 3. Create a synthetic financial-crime dataset

In [5]:
documents = [
    "A bank may flag international payments inconsistent with a customer's normal activity.",
    "Structuring splits transactions into smaller amounts to avoid reporting or monitoring thresholds.",
    "Correspondent banking lets one financial institution provide services to another institution.",
    "A sudden increase in cross-border transfers can warrant review against expected customer activity.",
    "Sanctions screening checks customers and payment parties against applicable sanctions lists.",
    "A false positive is an alert reviewed and found not to represent suspicious activity.",
    "Entity resolution helps determine whether records refer to the same person or organization.",
    "Trade-based money laundering can misuse invoices, goods, or trade transactions to move illicit value.",
    "Transaction monitoring applies rules or models to identify activity that may need investigation.",
    "Know Your Customer procedures collect and verify customer information and expected activity.",
    "Unusual payment corridors may be relevant when assessing the risk of an international transfer.",
    "An investigator should document evidence and rationale supporting an alert disposition."
]
ids = [f"doc_{i:02d}" for i in range(1, len(documents) + 1)]
topics = [
    "cross_border", "structuring", "correspondent_banking", "cross_border",
    "sanctions", "alert_quality", "entity_resolution", "trade_based_ml",
    "monitoring", "kyc", "cross_border", "investigation"
]
metadatas = [{"topic": topic, "source": "synthetic_training_data"} for topic in topics]

In [7]:
dataset = pd.DataFrame({"id": ids, "document": documents, "topic": topics})


In [8]:
dataset

,id,document,topic
0,doc_01,A bank may flag international payments inconsi...,cross_border
1,doc_02,Structuring splits transactions into smaller a...,structuring
2,doc_03,Correspondent banking lets one financial insti...,correspondent_banking
3,doc_04,A sudden increase in cross-border transfers ca...,cross_border
4,doc_05,Sanctions screening checks customers and payme...,sanctions
5,doc_06,A false positive is an alert reviewed and foun...,alert_quality
6,doc_07,Entity resolution helps determine whether reco...,entity_resolution
7,doc_08,Trade-based money laundering can misuse invoic...,trade_based_ml
8,doc_09,Transaction monitoring applies rules or models...,monitoring
9,doc_10,Know Your Customer procedures collect and veri...,kyc


# 4. Generate embeddings once

In [9]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
document_embeddings = model.encode(documents)

# 5. BUILD a FAISS index
IndexFlatIP performs exact inner-product search. Since vectors are normalized, ranking corresponds to cosine similarity.

In [10]:
dimension = document_embeddings.shape[1]

In [11]:
dimension

384

In [12]:
faiss_index = faiss.IndexFlatIP(dimension)

In [13]:
faiss_index

<faiss.swigfaiss.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x10c520630> >

In [14]:
faiss_index.add(document_embeddings)

In [16]:
print("FAISS vectors:", faiss_index.ntotal, "| dimensions:", faiss_index.d)


FAISS vectors: 12 | dimensions: 384


# 6. Build a ChromaDB collection

This uses an in-memory client for the notebook session. ChromaDB is configured for cosine distance; it stores IDs, documents, metadata, and embeddings together.

In [17]:
chroma_client = chromadb.Client()
collection_name = "Chapter_50_vector_benchmare"
chroma_collection = chroma_client.create_collection(name = collection_name, metadata={"hnsw:space" : "cosine"})

In [20]:
chroma_collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=document_embeddings
)

In [21]:
chroma_collection.count()

12

In [22]:
assert chroma_collection.count() == len(documents)

# 7. Create comparable search functions 
FAISS returns inner-product similarity scores (higher is better). ChromaDB returns cosine distances (lower is better). Compare ranking and content, not the raw score numbers as if they had the same meaning.

In [37]:
def search_faiss(query : str, k : int = 3) -> list[dict]:
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a non empty string")
    if not isinstance(k, int) or not 1<= k <= len(documents):
        raise ValueError(f"k mjst be an integer from 1 to {len(documents)}")

    query_vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")    
    scores, positions = faiss_index.search(query_vector, k)

    results = []

    for score, position in zip(scores[0], positions[0]):
        if position == -1:
            continue
        position = int(position)
        results.append({
            "id" : ids[position], "document" :documents[position],
            "topic" : topics[position], "score" : float(score),
            "score_type" : "inner_product_similarity"
        })
    return results


In [40]:
def search_chroma(query:str, k: int =3) -> list[dict]:
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a non empty string")
    if not isinstance(k, int) or not 1 <= k <= len(documents):
        raise ValueError(f"k must be an integer from 1 to {len(documents)}")

    query_vector = model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")

    response = chroma_collection.query(query_embeddings = query_vector,
                                       n_results = k) 
    return [
        {
            "id": result_id,
            "document": response["documents"][0][i],
            "topic": response["metadatas"][0][i]["topic"],
            "score": float(response["distances"][0][i]),
            "score_type": "cosine_distance"
        }
        for i, result_id in enumerate(response["ids"][0])
    ]

# 8. Compare one query

In [41]:
query = "How can a bank identify unusual international transfers?"
faiss_results = search_faiss(query, k=3)
chroma_results = search_chroma(query, k=3)

In [46]:
print(query)
print("\\nFAISS — higher similarity is better:")

for rank, item in enumerate(faiss_results, start =1):
    print(f"{rank}. {item['id']} | similarity={item['score']:.4f} | {item['document']}")


How can a bank identify unusual international transfers?
\nFAISS — higher similarity is better:
1. doc_01 | similarity=0.6486 | A bank may flag international payments inconsistent with a customer's normal activity.
2. doc_11 | similarity=0.6296 | Unusual payment corridors may be relevant when assessing the risk of an international transfer.
3. doc_08 | similarity=0.4607 | Trade-based money laundering can misuse invoices, goods, or trade transactions to move illicit value.


In [47]:
print("\\nChromaDB — lower cosine distance is better:")
for rank, item in enumerate(chroma_results, start=1):
    print(f"{rank}. {item['id']} | distance={item['score']:.4f} | {item['document']}")

\nChromaDB — lower cosine distance is better:
1. doc_01 | distance=0.3514 | A bank may flag international payments inconsistent with a customer's normal activity.
2. doc_11 | distance=0.3704 | Unusual payment corridors may be relevant when assessing the risk of an international transfer.
3. doc_08 | distance=0.5393 | Trade-based money laundering can misuse invoices, goods, or trade transactions to move illicit value.


In [48]:
f_ids = [r["id"] for r in faiss_results]
c_ids = [r["id"] for r in chroma_results]
print("\\nSame top-3 order?", f_ids == c_ids)
print("Top-3 overlap:", len(set(f_ids) & set(c_ids)), "of 3")

\nSame top-3 order? True
Top-3 overlap: 3 of 3
